# MAFCA — Multi-Agent Fact-Checking Assistant

Reproduction notebook for *A Multi-Agent Fact-Checking Assistant*.

Runs the three-agent pipeline against a single-LLM baseline on AVeriTeC and
FEVER, scores verdict accuracy, calibration, retrieval quality and citation
quality, and replays the aggregation ablation.

**What this notebook does and does not reproduce.** The aggregation ablation
(Section 4.8 of the paper) replays stored audits and is *exactly*
reproducible. The live-retrieval runs are not: DuckDuckGo returns different
documents from one day to the next, so re-running the benchmark will not
reproduce the paper's numbers to the decimal. The seeded claim sample is
identical, and the direction of the comparison is what the paper claims, not
the exact values.

Each section below is one runnable cell group.

## 1 · Get the code

In [ ]:
import os, pathlib, subprocess, sys

IN_COLAB = "google.colab" in sys.modules

# Point this at your copy if you are running from a fork or a local clone.
REPO_URL = os.environ.get("MAFCA_REPO", "")
PROJECT = pathlib.Path("/content/mafca") if IN_COLAB else pathlib.Path.cwd()

if IN_COLAB and not PROJECT.exists():
    if not REPO_URL:
        raise SystemExit(
            "Set MAFCA_REPO to your repository URL, or upload the project "
            "folder to /content/mafca before running this cell."
        )
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT)], check=True)

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))
print("working in", PROJECT)

## 2 · Install dependencies

In [ ]:
%pip install -q -r requirements.txt

## 3 · Choose a model backend

Two options, and the pipeline code is identical either way.

* **Ollama (local, free).** This is what the paper used — `qwen2.5:3b`, the
  smallest configuration the implementation supports. On Colab the install
  below takes a few minutes and needs a GPU runtime to be tolerably fast.
* **Anthropic (hosted, paid).** Set `ANTHROPIC_API_KEY`. Faster and stronger,
  but it costs money and will *not* reproduce the paper's 3B numbers.

Run **one** of the next two cells.

In [ ]:
# --- Option A: local Ollama (the paper's configuration) --------------------
import os, subprocess, time, urllib.request

MODEL = "qwen2.5:3b"

if IN_COLAB:
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(60):  # wait for the server rather than guessing at a delay
        try:
            urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
            break
        except Exception:
            time.sleep(2)
    subprocess.run(["ollama", "pull", MODEL], check=True)

os.environ["FACTCHECK_PROVIDER"] = "ollama"
os.environ["FACTCHECK_MODEL"] = MODEL
os.environ["FACTCHECK_NUM_CTX"] = "8192"
# A 3B model holds a short auditor prompt far better than a long one.
os.environ["FACTCHECK_RESULTS_PER_QUERY"] = "3"
os.environ["FACTCHECK_MAX_EVIDENCE"] = "4"

from fact_checker.config import ollama_status
reachable, installed = ollama_status()
print("ollama reachable:", reachable, "| models:", installed)

In [ ]:
# --- Option B: hosted Anthropic API ----------------------------------------
# import os, getpass
# os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")
# os.environ["FACTCHECK_PROVIDER"] = "anthropic"
# os.environ["FACTCHECK_MODEL"] = "claude-opus-5"

## 4 · Run the test suite

Everything below depends on these passing. They use no network and no model.

In [ ]:
!python -m pytest tests/ -q

## 5 · Fetch the benchmark splits

AVeriTeC's dev split is ~1.8 MB; FEVER's is larger. Both are cached under
`data/`, so this is a one-off.

In [ ]:
!python src/evaluate.py --dataset averitec --download
!python src/evaluate.py --dataset fever --download

## 6 · Preview the sample before paying for it

`--dry-run` shows the seeded, label-stratified claim set and the cost
estimate without calling a model.

In [ ]:
!python src/evaluate.py --dataset averitec --n 20 --seed 20240101 --dry-run

## 7 · Main run — AVeriTeC, n = 60

20 claims per label. On a 3B local model this takes roughly 15–25 minutes
with three workers; most of that is web-search round trips, not inference.

Results stream to JSONL as they land, so an interrupted run resumes with
`--resume` rather than repeating completed claims.

Lower `--n` to 3 or 5 for a quick smoke run.

In [ ]:
!python src/evaluate.py --dataset averitec --n 20 --seed 20240101 \
    --workers 3 --out results/averitec_colab.jsonl --yes

## 8 · Secondary run — FEVER, n = 30

FEVER is the comparability corpus. Read its NOT_ENOUGH_INFO numbers with the
caveat the harness prints: FEVER's NEI means "unverifiable from a 2017
Wikipedia snapshot", which an open-web system does not implement.

In [ ]:
!python src/evaluate.py --dataset fever --n 10 --seed 20240101 \
    --workers 3 --out results/fever_colab.jsonl --yes

## 9 · Aggregation ablation (free, exact)

This replays aggregation policies over the audits stored by the run above.
It performs no inference, so it is instant — and because retrieval and
verification are held fixed, any difference between rows is attributable to
the aggregation function alone.

`beta` is the specificity weight of Eq. (18); `beta = 0` is unweighted
voting.

In [ ]:
!python src/analysis/ablation.py --results results/averitec_colab.jsonl \
    --tag av --betas 0 1 2

## 10 · Oracle retrieval (upper bound)

Substitutes the evidence a human fact-checker actually used for whatever
search returns, which scores verification quality independently of
retrieval quality. It is an upper bound, not a deployable mode: at inference
time no gold evidence exists.

In [ ]:
!python src/evaluate.py --dataset averitec --n 20 --seed 20240101 --oracle \
    --workers 3 --out results/averitec_colab_oracle.jsonl --yes

## 11 · Figures and tables

Every value is computed from the results file; nothing is hardcoded, so a
chart cannot drift away from the run it describes.

In [ ]:
!python src/analysis/results_report.py --results results/averitec_colab.jsonl --tag av
!python src/analysis/results_report.py --results results/fever_colab.jsonl --tag fev

In [ ]:
import pathlib
from IPython.display import Image, display

for name in ["av_accuracy", "av_f1", "av_latency", "av_errorrate"]:
    path = pathlib.Path("results/figures") / f"{name}.png"
    if path.exists():
        print(name)
        display(Image(filename=str(path)))

## 12 · Check a single claim interactively

The full trace, stage by stage. `exclude_url` is left empty here: outside the
benchmark harness there is no originating fact-check article to keep out.

In [ ]:
from fact_checker import Settings
from fact_checker.pipeline import run_multi_agent
from fact_checker.agents import run_baseline

CLAIM = "The Eiffel Tower was completed in 1889."

settings = Settings(deterministic_aggregation=True)
result = run_multi_agent(CLAIM, settings, on_progress=print)

print("\nqueries:", result.scout.queries)
for item in result.scout.evidence:
    print(f"  {item.source_id}  {item.url}")
print("\nper-passage audit:")
for a in result.audit.alignments:
    print(f"  {a.source_id}  {a.verdict:<16} {a.justification}")
if result.aggregation is not None:
    print("\naggregation:", result.aggregation.explain())
print("\nverdict   :", result.adjudication.final_verdict,
      f"(confidence {result.adjudication.confidence:.2f})")
print("citations :", result.adjudication.cited_sources)

baseline = run_baseline(CLAIM, settings)
print("\nbaseline  :", baseline.adjudication.final_verdict,
      f"(confidence {baseline.adjudication.confidence:.2f}) — no evidence")

## 13 · Save the outputs

On Colab the runtime is wiped when the session ends.

In [ ]:
import shutil

archive = shutil.make_archive("mafca_outputs", "zip", base_dir="results")
print("wrote", archive)

if IN_COLAB:
    from google.colab import files
    files.download(archive)